# 07 — Pairs, splits and the information firewall

**Stage question.** What is one unit of analysis, are the splits session-level and speaker-disjoint, and does the per-pair firewall hold in both directions?

| | |
|---|---|
| **Purpose** | Build the speaker-disjoint 70/10/20 split of sessions (and the dyad-disjoint contrast unless the settings drop it), check it, and show the firewall refusing a pair's own reply while serving the same turn as another pair's partner. |
| **Input** | `turns.parquet`, `pairs_candidates.parquet` (notebook 02); `selected_sessions_<stage>.csv` (notebook 03); the manifest; raw transcripts. |
| **Processing** | speaker graph + transcript near-duplicate links → groups → seeded greedy assignment → disjointness assertion → split report; firewall demonstration on real pairs. |
| **Output** | `manifests/splits/split__<stage>__<scheme>__<hash>.csv` and `.json` (tracked). |
| **Scientific assumptions** | Participant identifiers identify people (checked, not assumed); one session per speaker. |
| **Failure modes** | A speaker on both sides of a split (assertion raises); near-duplicates split apart (assertion raises); a degenerate contrast (reported). |
| **What to look at** | The split report and the shared-speaker count of the contrast. |

**Terms used in this notebook.**

- **Session** — one dyad, about an hour; the unit of analysis, resampling and splitting.
- **Speaker-disjoint** — no participant appears in more than one partition.
- **Firewall** — the rule that a pair's own reply audio, duration or delivery is never an input for that pair.

> **Before running.** Uses the near-duplicate rule and the contrast-split setting from `src/settings.py`.

In [2]:
# Runtime setup — identical in every notebook.
# Colab: mounts Google Drive and uses MyDrive/coda (a copy of the repository) as
#        the project root, then installs the pinned measurement stack.
# Local: uses the repository checkout.
# Every path used below is relative to that root, so Drive and the repository
# have exactly the same layout and outputs can be copied back unchanged.
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    ROOT = Path(os.environ.get('CODA_ROOT', '/content/drive/MyDrive/coda'))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(ROOT / 'requirements-colab.txt')], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

os.environ['CODA_ROOT'] = str(ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.runtime import runtime_report
REPORT = runtime_report()

Mounted at /content/drive
runtime        colab   python 3.13.16   cpus 2
memory         12.3 GB free of 13.6 GB
gpu            none (not needed for this stage unless stated)
project disk   11.8 GB free   root = /content/drive/MyDrive/coda
scratch disk   93.4 GB free   (audio lives here briefly)
code           fingerprint e1a10dc2c7145664
artifacts      manifest=True, metadata=True, transcripts=True, turns=True, audio_pass_sessions=293, channel_qc=True


**Runtime check.** The cell above prints free disk, memory, CPU, GPU, the code fingerprint and which artifacts already exist. The fingerprint identifies the exact source and configuration that ran and is written into every run record.

In [3]:
STAGE = 'sample'   # 'pilot' for development; 'sample' only after the preflight
STEPS = ['splits']

## Settings

The method settings come from `src/settings.py` (class `Settings`).

In [4]:
from src.settings import Settings
SETTINGS = Settings()

## 1. Split by session under the speaker-disjointness constraint

In [5]:
from src.paths import paths
from src.experiments.stages import splits
P = paths()
primary, rep = splits(P, STAGE, 'speaker_disjoint', SETTINGS)
rep

{'scheme': 'speaker_disjoint',
 'seed': 20260912,
 'sessions': {'train': 176, 'val': 25, 'test': 49},
 'speakers': {'train': 352, 'val': 50, 'test': 98},
 'groups_with_more_than_one_session': 4,
 'speakers_shared_across_partitions': 0,
 'manifest_hash': '19379138530a',
 'near_duplicate_links': [],
 'identity_candidate_links': [['V02_S3243', 'V02_S3289', '1.0'],
  ['V02_S3243', 'V02_S3293', '1.0'],
  ['V02_S3243', 'V02_S3317', '1.0'],
  ['V02_S3243', 'V02_S3347', '1.0'],
  ['V02_S3243', 'V02_S3442', '1.0'],
  ['V02_S3243', 'V02_S3446', '1.0'],
  ['V02_S3243', 'V02_S3452', '1.0'],
  ['V02_S3243', 'V02_S3590', '1.0'],
  ['V02_S3305', 'V02_S3389', '1.0'],
  ['V02_S3364', 'V02_S3456', '1.0'],
  ['V02_S3493', 'V02_S3655', '1.0']]}

**Reading it.** The report gives sessions, speakers and pairs per partition, the number of groups holding more than one session (sessions linked by a shared speaker, a near-duplicate transcript, or a metadata identity candidate), and the manifest hash every later run records. Anything but zero shared speakers would have raised before this cell printed.

In [6]:
print('identity-candidate links applied:', len(rep['identity_candidate_links']))
print('near-duplicate transcript links:', len(rep['near_duplicate_links']))

identity-candidate links applied: 11
near-duplicate transcript links: 0


**Reading it.** Identity-candidate links come from the pre-registered metadata screen; linked sessions share a split group and nothing is excluded. Disjointness holds by **recorded participant ID**, with these candidates grouped. It is not verified unseen-person separation.

## 2. The dyad-disjoint contrast

In [7]:
from src.experiments.stages import contrast_enabled
if contrast_enabled(SETTINGS):
    contrast, rep_c = splits(P, STAGE, 'dyad_disjoint', SETTINGS)
    print('speakers shared across partitions:',
          rep_c['speakers_shared_across_partitions'])
else:
    print('the dyad-disjoint contrast is dropped by the settings, so it is not built')

the dyad-disjoint contrast is dropped by the settings, so it is not built


**Reading it.** Every speaker belongs to one session in this sample, so the dyad-disjoint contrast cannot hold a speaker out of one partition while keeping them in another. A count of zero confirms that this contrast is a second random speaker-disjoint partition, not a seen-speaker contrast (audit finding A-1).

## 3. The firewall on real pairs

In [8]:
import pandas as pd

from src.data.dataset import PairFirewall, LeakageError
turns = pd.read_parquet(P.processed / 'turns.parquet')
pairs = pd.read_parquet(P.processed / 'pairs_candidates.parquet')
fw = PairFirewall(turns, pairs)
first, second = pairs.iloc[0], pairs.iloc[1]
try:
    fw.request(first['pair_id'], 'reply', 'duration')
except LeakageError as err:
    print('refused as intended:', err)
print('same turn as the next pair\'s partner:',
      fw.request(second['pair_id'], 'partner', 'duration'))

refused as intended: V00_S0051_I00000125_T0001__after: 'duration' of the pair's own reply is not an allowed input
same turn as the next pair's partner: 0.3800000000000523


**Reading it.** The first request asks for a pair's own reply duration and is refused. The second asks for the same turn as the partner of the following pair and is served. That is the per-pair rule, shown on the corpus rather than asserted.